# Answer Comparison Parser — Bug Report

`answer_comparison_extract()` in `experiment_result_table.ipynb` contains two regex bugs that cause systematically wrong accuracy scores. Both affect all paper models.

---

In [1]:
import re, pandas as pd
from glob import glob
from pathlib import Path
from sklearn.metrics import accuracy_score, f1_score

## Bug 1 — Pattern 1 matches body text, not the verdict

```python
pattern = r"(?:answer\s*)?([ab]).*?correct"  # FIRES FIRST on ~92% of rows
```

This pattern fires whenever any `a` or `b` character appears before the word `correct` anywhere in the response. For responses that open with phrasing like `"Both answers A and B..."` or `"analyze both..."`, it captures a letter from the body rather than the verdict.

In [2]:
cases = [
    ("Both answers A and B discuss the topic. Answer A is correct.",  "A"),
    ("Let's analyze both answers. Answer B is factually correct.",    "B"),
    ("Kevlar is a fabric which spreads impact. Answer B is correct.", "B"),
]
for text, expected in cases:
    m = re.search(r"(?:answer\s*)?([ab]).*?correct", text, re.IGNORECASE)
    print(f"  captured={m.group(1).upper()!r}  expected={expected!r}  {'OK' if m.group(1).upper()==expected else 'WRONG'}")
    print(f"  match: {m.group()!r}")

  captured='B'  expected='A'  WRONG
  match: 'Both answers A and B discuss the topic. Answer A is correct'
  captured='A'  expected='B'  WRONG
  match: 'analyze both answers. Answer B is factually correct'
  captured='A'  expected='B'  WRONG
  match: 'ar is a fabric which spreads impact. Answer B is correct'


## Bug 2 — Pattern 2 lazy quantifier captures 'A' from 'Answer B'

```python
pattern = r"\**\s*Final\s+Verdict.*?(A|B)"   # lazy .*? stops too early
```

`.*?` matches as few characters as possible before handing off to `(A|B)`. When the LLM writes `"Final Verdict: Answer B"`, the engine stops at the `A` in `Answer` instead of continuing to the verdict letter `B`.

Fix: `r"\**\s*Final\s+Verdict.*?\b(A|B)\b"` — word boundaries ensure standalone letter only.

In [3]:
example = "**Final Verdict: Answer B**"
buggy = re.search(r"\**\s*Final\s+Verdict.*?(A|B)",       example, re.IGNORECASE)
fixed = re.search(r"\**\s*Final\s+Verdict.*?\b(A|B)\b",   example, re.IGNORECASE)
print(f"Input:  {example!r}")
print(f"Buggy:  captures {buggy.group(1)!r}  (matched {buggy.group()!r})")
print(f"Fixed:  captures {fixed.group(1)!r}  (matched {fixed.group()!r})")

Input:  '**Final Verdict: Answer B**'
Buggy:  captures 'A'  (matched '**Final Verdict: A')
Fixed:  captures 'B'  (matched '**Final Verdict: Answer B')


## Corrected Accuracy — All Paper Models

The robust parser replaces both broken patterns with a single reliable approach: find the `Final Verdict` line, extract the first standalone `A` or `B` using word boundaries.

In [8]:
def buggy_parser(text):
    if not isinstance(text, str): return None
    m = re.search(r"(?:answer\s*)?([ab]).*?correct", text, re.IGNORECASE)
    if m: return m.group(1).upper()
    m = re.search(r"\**\s*Final\s+Verdict.*?(A|B)", text, re.IGNORECASE)
    if m: return m.group(1).upper()
    return None

def robust_parser(text):
    if not isinstance(text, str): return None
    m = re.search(r"final verdict[^\n]*", text, re.IGNORECASE)
    if m:
        letter = re.search(r"\b(A|B)\b", m.group(), re.IGNORECASE)
        if letter: return letter.group(1).upper()
    return None

def y_true(gt): return "A" if gt[0] == "Original" else "B"

models = [
    "gemma-3-1b-it",
    "gemma-3-4b-it",
    "gemma-3-12b-it",
    "gemma-3-27b-it",
    "Llama-3.2-1B-Instruct",
    "Llama-3.2-3B-Instruct",
    "Llama-3.3-70B-Instruct",
]
default_filename = "llm_outputs/knowledge/results_answer_comparison__{}__zeroshot__no_self_consistency.json"
files = [default_filename.format(model) for model in models]
rows = []
for path in files:
    model = Path(path).name.split("__")[1]
    df = pd.read_json(path)
    df["true"]        = df["ground_truth"].apply(y_true)
    df["pred_buggy"]  = df["output"].apply(buggy_parser)
    df["pred_robust"] = df["output"].apply(robust_parser)
    rows.append({
        "model":        model,
        "acc_buggy":    round(accuracy_score(df["true"], df["pred_buggy"].fillna("_")),  2),
        "f1_buggy":     round(f1_score(      df["true"], df["pred_buggy"].fillna("_"),  average="macro", zero_division=0), 2),
        "acc_robust":   round(accuracy_score(df["true"], df["pred_robust"].fillna("_")), 2),
        "f1_robust":    round(f1_score(      df["true"], df["pred_robust"].fillna("_"), average="macro", zero_division=0), 2),
        "delta_acc":    round(accuracy_score(df["true"], df["pred_robust"].fillna("_")) -
                              accuracy_score(df["true"], df["pred_buggy"].fillna("_")),  2),
        "parse_fail":   int(df["pred_robust"].isna().sum()),
    })

results = pd.DataFrame(rows).set_index("model")
print(results.to_string())

                        acc_buggy  f1_buggy  acc_robust  f1_robust  delta_acc  parse_fail
model                                                                                    
gemma-3-1b-it                0.53      0.34        0.50       0.34      -0.03          74
gemma-3-4b-it                0.52      0.34        0.53       0.36       0.01         130
gemma-3-12b-it               0.71      0.47        0.48       0.39      -0.23         377
gemma-3-27b-it               0.55      0.54        0.84       0.56       0.28          15
Llama-3.2-1B-Instruct        0.50      0.27        0.22       0.21      -0.27         580
Llama-3.2-3B-Instruct        0.48      0.29        0.22       0.20      -0.26         602
Llama-3.3-70B-Instruct       0.50      0.39        0.88       0.59       0.38           7


## Corrected Accuracy Ignoring Rows Without a "Final Verdict" Line

In [9]:
rows_cond = []
for path in files:
    model = Path(path).name.split("__")[1]
    df = pd.read_json(path)
    df["true"]        = df["ground_truth"].apply(y_true)
    df["pred_robust"] = df["output"].apply(robust_parser)
    df["pred_buggy"]  = df["output"].apply(buggy_parser)
    parseable = df[df["pred_robust"].notna()]
    rows_cond.append({
        "model":            model,
        "acc_robust_cond":  round(accuracy_score(parseable["true"], parseable["pred_robust"]), 2),
        "acc_buggy_cond":   round(accuracy_score(parseable["true"], parseable["pred_buggy"]), 2),
        "parseable":        len(parseable),
        "total":            len(df),
    })

cond = pd.DataFrame(rows_cond).set_index("model")
print(cond.to_string())

                        acc_robust_cond  acc_buggy_cond  parseable  total
model                                                                    
gemma-3-1b-it                      0.54            0.53        927   1001
gemma-3-4b-it                      0.61            0.52        871   1001
gemma-3-12b-it                     0.77            0.64        624   1001
gemma-3-27b-it                     0.85            0.55        986   1001
Llama-3.2-1B-Instruct              0.53            0.51        421   1001
Llama-3.2-3B-Instruct              0.54            0.49        399   1001
Llama-3.3-70B-Instruct             0.89            0.50        994   1001


## Unstructured Responses Make Parsing Difficult

In [10]:
llama_3b_results_path = default_filename.format("Llama-3.2-3B-Instruct")
llama_3b_df = pd.read_json(llama_3b_results_path).set_index("id")
llama_3b_df["pred_robust"] = llama_3b_df["output"].apply(robust_parser)
bad_output_df = llama_3b_df[llama_3b_df["pred_robust"].isna()]
bad_samples = bad_output_df.sample(3, random_state=42)
bad_sample_list = list(bad_samples["output"])
print("\n======================\n".join(bad_sample_list))

After analyzing both answers, I conclude that Answer B is factually correct.

While it is true that the sun's position does wobble north and south through the year, and there are monsoon patterns in the equatorial regions, the primary factor that influences tree growth at the equator is indeed the wet and dry seasons. These seasonal variations in rainfall and water availability still lead to differences in tree growth, resulting in distinct rings. The fact that the sun is directly overhead during both wet and dry seasons does not negate the impact of the seasonal changes on tree growth.

Answer A's mention of solar cycles is also accurate, as solar cycles can have an impact on tree growth, but it is not the primary factor at the equator. The mention of monsoon patterns in Answer A is also correct, but it is not the primary factor that distinguishes the growth rings at the equator.

Therefore, the correct answer is B, as it accurately highlights the impact of both wet and dry seasons on

## Summary

### Two bugs, one parser

```python
# Bug 1 — fires on ~92% of rows; captures random a/b from body text
pattern = r"(?:answer\s*)?([ab]).*?correct"

# Bug 2 — lazy .*? captures 'A' from 'Answer B'
pattern = r"\**\s*Final\s+Verdict.*?(A|B)"

# Robust replacement for both:
m = re.search(r"final verdict[^\n]*", text, re.IGNORECASE)
letter = re.search(r"\b(A|B)\b", m.group(), re.IGNORECASE)
```

### Interpretation of the results table

- **Llama-3.3-70B** and **gemma-3-27b**: these models consistently write `"Final Verdict: Answer X"` — the robust parser resolves nearly all rows and reveals their buggy accuracy (~0.50–0.55) was severely underestimated. True accuracy is **0.88** and **0.84** respectively.
- For the rest, true accuracy is hard to judge as many answers lack a clear, parsable format.

### Bottom line

Reported results are severely inaccurate due to parsing errors. We would probably need a separate LLM to extract the A / B answer flags from the unstructured outputs. The cleanest fix is to re-run the experiments with a prompt that instructs the LLMs to use a structured, parseable output format.